# Module 6 - Outlier Treatment

## 1. What is an Outlier?

An outlier is a data point that is significantly different from most of the other observations in a dataset.

Outliers can occur due to data entry errors, measurement problems, unusual events, or genuine extreme observations.

Outliers should be investigated before deciding whether to remove, cap, transform, or retain them.

In [2]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv(r"C:\Users\HP\sprint-5-data-cleaning-preprocessing\data\hotel_bookings.csv")

# Display the first five records
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
numeric_columns = df.select_dtypes(include=['number']).columns

print("Numerical Columns:")
print(numeric_columns.tolist())

print("\nSummary Statistics:")
print(df[numeric_columns].describe())

Numerical Columns:
['is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Summary Statistics:
         is_canceled      lead_time  arrival_date_year  \
count  119390.000000  119390.000000      119390.000000   
mean        0.370416     104.011416        2016.156554   
std         0.482918     106.863097           0.707476   
min         0.000000       0.000000        2015.000000   
25%         0.000000      18.000000        2016.000000   
50%         0.000000      69.000000        2016.000000   
75%         1.000000     160.000000        2017.000000   
max         1.000000     737.000000        2017.000000   

       arrival_date_week_number  arr

### Observation

The numerical columns were examined using summary statistics to understand their distributions and identify values that may be unusually high or low compared with the other observations.

## 2. Outlier vs Error

An outlier is a value that is unusually different from the other observations, but it may still be a valid observation.

An error is an incorrect value caused by data entry mistakes, measurement problems, or data processing issues.

For example, an unusually high `adr` may be a valid booking with a premium room, while a negative `adr` may indicate a data error.

Therefore, every outlier should be investigated before deciding how to handle it.

In [4]:
print("ADR Summary:")
print(df['adr'].describe())

print("\nMinimum ADR:", df['adr'].min())
print("Maximum ADR:", df['adr'].max())

print("\nRecords with negative ADR:")
print(df[df['adr'] < 0][['hotel', 'adr']].head())

ADR Summary:
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

Minimum ADR: -6.38
Maximum ADR: 5400.0

Records with negative ADR:
              hotel   adr
14969  Resort Hotel -6.38


### Observation

The `adr` column was examined for unusually high, low, and negative values. An outlier is not automatically an error, so the detected values should be investigated based on their business meaning before deciding whether to remove, cap, or retain them.

## 3. IQR Method

The Interquartile Range (IQR) method is a statistical technique used to identify outliers.

IQR is calculated as:

IQR = Q3 - Q1

The lower bound is:

Q1 - 1.5 × IQR

The upper bound is:

Q3 + 1.5 × IQR

Values below the lower bound or above the upper bound are considered potential outliers.

For the hotel bookings dataset, the IQR method can be applied to numerical columns such as `adr`.

In [5]:
Q1 = df['adr'].quantile(0.25)
Q3 = df['adr'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['adr'] < lower_bound) |
    (df['adr'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Number of potential outliers:", len(outliers))

Q1: 69.29
Q3: 126.0
IQR: 56.709999999999994
Lower Bound: -15.774999999999991
Upper Bound: 211.065
Number of potential outliers: 3793


### Observation

The IQR method was applied to the `adr` column to identify potential outliers. Values outside the calculated lower and upper bounds were identified for further investigation.

## 4. Z-Score Method

The Z-Score method identifies outliers by measuring how far a value is from the mean in terms of standard deviations.

A Z-score shows the relative distance of a value from the mean.

A common rule is that values with an absolute Z-score greater than 3 are considered potential outliers.

For the hotel bookings dataset, the Z-Score method can be applied to numerical columns such as `adr`.

In [7]:
from scipy.stats import zscore

adr_zscore = zscore(df['adr'].dropna())

outliers = df.loc[
    df['adr'].dropna().index[
        abs(adr_zscore) > 3
    ]
]

print("Number of potential outliers:", len(outliers))
print("\nPotential outliers:")
print(outliers[['hotel', 'adr']].head(10))

Number of potential outliers: 1138

Potential outliers:
             hotel     adr
803   Resort Hotel  280.74
925   Resort Hotel  268.00
936   Resort Hotel  267.00
943   Resort Hotel  277.50
973   Resort Hotel  276.43
998   Resort Hotel  277.00
1008  Resort Hotel  254.00
1047  Resort Hotel  274.93
1063  Resort Hotel  258.33
1081  Resort Hotel  255.00


### Observation

The Z-Score method was applied to the `adr` column. Values with an absolute Z-score greater than 3 were identified as potential outliers for further investigation.

## 5. Percentile Method

The Percentile method identifies extreme values by defining lower and upper percentile limits.

For example, values below the 1st percentile or above the 99th percentile can be considered potential outliers.

This method is useful when the data contains extreme values and a fixed percentage of the lowest and highest observations needs to be identified.

For the hotel bookings dataset, the Percentile method can be applied to the `adr` column.

In [8]:
lower_percentile = df['adr'].quantile(0.01)
upper_percentile = df['adr'].quantile(0.99)

outliers = df[
    (df['adr'] < lower_percentile) |
    (df['adr'] > upper_percentile)
]

print("1st Percentile:", lower_percentile)
print("99th Percentile:", upper_percentile)
print("Number of potential outliers:", len(outliers))

print("\nPotential outliers:")
print(outliers[['hotel', 'adr']].head(10))

1st Percentile: 0.0
99th Percentile: 252.0
Number of potential outliers: 1169

Potential outliers:
             hotel     adr
803   Resort Hotel  280.74
925   Resort Hotel  268.00
936   Resort Hotel  267.00
943   Resort Hotel  277.50
973   Resort Hotel  276.43
998   Resort Hotel  277.00
1008  Resort Hotel  254.00
1047  Resort Hotel  274.93
1063  Resort Hotel  258.33
1081  Resort Hotel  255.00


### Observation

The Percentile method was applied to the `adr` column. Values below the 1st percentile and above the 99th percentile were identified as potential outliers for further investigation.

## 6. Winsorization

Winsorization is an outlier treatment technique that limits extreme values to predefined percentile boundaries instead of removing them.

For example, values below the 1st percentile can be replaced with the 1st percentile value, and values above the 99th percentile can be replaced with the 99th percentile value.

Winsorization is useful when extreme values may contain useful information but can have a strong effect on statistical analysis.

In [9]:
lower_limit = df['adr'].quantile(0.01)
upper_limit = df['adr'].quantile(0.99)

df_winsorized = df.copy()

df_winsorized['adr'] = df_winsorized['adr'].clip(
    lower=lower_limit,
    upper=upper_limit
)

print("Original ADR minimum:", df['adr'].min())
print("Original ADR maximum:", df['adr'].max())

print("\nWinsorized ADR minimum:", df_winsorized['adr'].min())
print("Winsorized ADR maximum:", df_winsorized['adr'].max())

Original ADR minimum: -6.38
Original ADR maximum: 5400.0

Winsorized ADR minimum: 0.0
Winsorized ADR maximum: 252.0


### Observation

Extreme values in the `adr` column were limited to the 1st and 99th percentile boundaries. The records were retained, but their extreme values were reduced to prevent them from having an excessive influence on the analysis.

## 7. Capping

Capping is an outlier treatment technique in which extreme values are replaced with predefined upper and lower limits.

Unlike removing outliers, capping keeps all records in the dataset while limiting the effect of extreme values.

For the hotel bookings dataset, capping can be applied to the `adr` column using the IQR lower and upper bounds.

In [10]:
Q1 = df['adr'].quantile(0.25)
Q3 = df['adr'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_capped = df.copy()

df_capped['adr'] = df_capped['adr'].clip(
    lower=lower_bound,
    upper=upper_bound
)

print("Lower Capping Limit:", lower_bound)
print("Upper Capping Limit:", upper_bound)

print("\nOriginal ADR Range:")
print(df['adr'].min(), "to", df['adr'].max())

print("\nCapped ADR Range:")
print(df_capped['adr'].min(), "to", df_capped['adr'].max())

Lower Capping Limit: -15.774999999999991
Upper Capping Limit: 211.065

Original ADR Range:
-6.38 to 5400.0

Capped ADR Range:
-6.38 to 211.065


### Observation

The extreme values in the `adr` column were capped using the IQR boundaries. The records were retained while limiting the influence of extreme values on further analysis.

## 8. Flooring

Flooring is an outlier treatment technique in which values below a predefined lower limit are replaced with that lower limit.

It is useful when very low values are invalid or when extreme low values need to be controlled without removing the records.

For the hotel bookings dataset, flooring can be applied to the `adr` column by setting a lower limit.

In [11]:
lower_limit = df['adr'].quantile(0.01)

df_floored = df.copy()

df_floored['adr'] = df_floored['adr'].clip(
    lower=lower_limit
)

print("Flooring Limit:", lower_limit)

print("\nOriginal ADR minimum:", df['adr'].min())
print("Floored ADR minimum:", df_floored['adr'].min())

Flooring Limit: 0.0

Original ADR minimum: -6.38
Floored ADR minimum: 0.0


### Observation

Values below the selected lower percentile were replaced with the flooring limit. The records were retained while reducing the effect of extremely low `adr` values.

## 9. Transformation

Transformation is a technique used to reduce the effect of extreme values and make a numerical variable more suitable for analysis.

Common transformations include logarithmic, square root, and power transformations.

For the hotel bookings dataset, a logarithmic transformation can be applied to `adr`. Since `adr` contains negative values, the values are first shifted to make them positive.

In [12]:
import numpy as np

df_transformed = df.copy()

shift = 1 - df_transformed['adr'].min()

df_transformed['adr_log'] = np.log1p(
    df_transformed['adr'] + shift
)

print("Original ADR:")
print(df['adr'].describe())

print("\nTransformed ADR:")
print(df_transformed['adr_log'].describe())

Original ADR:
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

Transformed ADR:
count    119390.000000
mean          4.592911
std           0.520147
min           0.693147
25%           4.352469
50%           4.634292
75%           4.900672
max           8.595705
Name: adr_log, dtype: float64


### Observation

A logarithmic transformation was applied to the `adr` column after shifting the values to handle negative observations. The transformation reduces the influence of extreme values while retaining the records for further analysis.

## 10. Removing Outliers

Removing outliers means deleting records that are identified as extreme or invalid observations.

Outliers should only be removed when there is a valid reason, such as a data entry error or an impossible value.

For the hotel bookings dataset, negative `adr` values can be considered invalid because Average Daily Rate should not be negative.

In [13]:
df_removed = df.copy()

invalid_adr = df_removed['adr'] < 0

print("Negative ADR records before removal:", invalid_adr.sum())

df_removed = df_removed[df_removed['adr'] >= 0]

print("Dataset shape before removal:", df.shape)
print("Dataset shape after removal:", df_removed.shape)
print("Records removed:", df.shape[0] - df_removed.shape[0])

Negative ADR records before removal: 1
Dataset shape before removal: (119390, 32)
Dataset shape after removal: (119389, 32)
Records removed: 1


### Observation

Negative `adr` values were treated as invalid observations and removed from the dataset. Other extreme `adr` values should not be removed automatically because they may represent valid hotel bookings.

## 11. Retaining Outliers

Retaining outliers means keeping extreme observations in the dataset when they represent valid and meaningful cases.

Not every outlier is an error. In the hotel bookings dataset, a very high `adr` may represent a genuine premium booking.

Outliers should be retained when they are valid observations and provide useful information for analysis.

In [14]:
Q1 = df['adr'].quantile(0.25)
Q3 = df['adr'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['adr'] < lower_bound) |
    (df['adr'] > upper_bound)
]

valid_extreme_values = outliers[outliers['adr'] >= 0]

print("Total potential outliers:", len(outliers))
print("Non-negative extreme values retained:", len(valid_extreme_values))
print("\nSample retained extreme values:")
print(valid_extreme_values[['hotel', 'adr']].head(10))

Total potential outliers: 3793
Non-negative extreme values retained: 3793

Sample retained extreme values:
            hotel     adr
140  Resort Hotel  225.00
303  Resort Hotel  213.75
396  Resort Hotel  230.67
412  Resort Hotel  216.13
523  Resort Hotel  249.00
526  Resort Hotel  241.50
558  Resort Hotel  214.00
580  Resort Hotel  214.00
584  Resort Hotel  240.64
632  Resort Hotel  217.05


### Observation

Potential outliers were identified using the IQR method. Non-negative extreme `adr` values were retained because they may represent valid hotel bookings rather than errors.

Outliers should be removed, capped, or transformed only after understanding their cause and business meaning.